# 05 — Anomaly & Exception Detection

Detect suspicious records using IsolationForest + rule-based flags. Classify exception types. Generate 25 reviewer-ready examples.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.modeling.anomaly import AnomalyDetector

train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
servicer_df = pd.read_csv('../data/synthetic/servicer_updates.csv')
for col in ['reporting_month','origination_month']:
    if col in train_df.columns:
        train_df[col] = pd.to_datetime(train_df[col]).dt.to_period('M')
from src.pipeline.loader import reconcile_servicer_updates
train_df, _ = reconcile_servicer_updates(train_df, servicer_df)
print(f'Train: {train_df.shape}')


## Load Trained Detector

In [ ]:
from src.utils.config import get_settings
config = get_settings('../config/settings.yaml')
detector = AnomalyDetector(config)
detector.load_artifacts('../models/anomaly')
print('Detector loaded')
result = detector.detect(train_df)
print(f'Flagged anomalies: {result.flags.sum()} / {len(train_df)} ({result.flags.mean():.1%})')
print(f'Score range: [{result.scores.min():.4f}, {result.scores.max():.4f}]  mean={result.scores.mean():.4f}')


## Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(result.scores, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(result.scores[result.flags==1].min(), color='red', linestyle='--', label='flag threshold')
axes[0].set_title('Anomaly Score Distribution')
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

status_counts = train_df.assign(flagged=result.flags).groupby(['current_status','flagged']).size().unstack(fill_value=0)
status_counts.plot(kind='bar', ax=axes[1], color=['#4C9BE8','#E87070'])
axes[1].set_title('Flagged vs Normal by Current Status')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../reports/anomaly_distribution.png', dpi=150)
plt.show()


## Reviewer-Ready Examples (top 10)

In [ ]:
examples = detector.generate_reviewer_examples(train_df, result, n_examples=25)
rows = []
for ex in examples[:10]:
    rows.append({
        'loan_id': ex.loan_id,
        'anomaly_score': round(ex.anomaly_score, 4),
        'exception_type': ex.exception_type,
        'suggested_action': ex.suggested_action,
        'top_driver': ex.top_drivers[0]['feature'] if ex.top_drivers else 'N/A',
        'rule_flags': ', '.join(ex.rule_flags[:2]) if ex.rule_flags else 'none'
    })
pd.DataFrame(rows)
